# 03 · A multi-agent team: PI + omics specialists (centralized architecture)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smusleh/LLM_Bioinformatics_Agents/blob/main/notebooks/03_multi_agent_pi.ipynb)

**Paper concepts:** *Multiple agents → Centralized architecture* (Fig. 3). One coordinator (here a "PI") breaks the problem down, delegates to specialists and combines their results, as in **The Virtual Lab**. There is one specialist agent per data type, as in **PromptBio** and **DrBioRight 2.0** (§ *Impact of omic data characteristics on agents collaboration*).

```
                         ┌──────────────┐
                         │   PI agent   │  plans · delegates · synthesises
                         └──────┬───────┘
        ┌───────────────┬───────┴───────┬──────────────────┐
 Transcriptomics      miRNA         Proteomics         Integration
 (mRNA only)       (miRNA only)    (protein only)   (early/late fusion,
  + KB search       + KB search     + KB search       held-out test)
```
Each specialist is **restricted** to its own data layer by its toolbox (see [`omics_agents/team.py`](../omics_agents/team.py)). Each one perceives only part of the data, so the team needs to communicate.

In [ ]:
# ▶ Colab setup: run this cell first (it does nothing when you run the notebook locally).
# BACKEND: "ollama"    free, no account; use Runtime -> Change runtime type -> T4 GPU (first run ~3-5 min)
#          "gemini"    free key from aistudio.google.com/apikey, saved as Colab secret GEMINI_API_KEY
#          "anthropic" Claude; Colab secret ANTHROPIC_API_KEY + API credits
BACKEND = "ollama"
import os, sys, subprocess
if "google.colab" in sys.modules:
    REPO_DIR = "/content/LLM_Bioinformatics_Agents"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "-q", "https://github.com/smusleh/LLM_Bioinformatics_Agents.git", REPO_DIR], check=True)
    os.chdir(f"{REPO_DIR}/notebooks")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "anthropic==2.8.0", "openai>=1.50.0", "rdata==0.11.2", "python-dotenv"], check=True)
    sys.path.insert(0, REPO_DIR)
    from omics_agents import colab
    colab.setup(BACKEND)

In [ ]:
import sys, os, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # make `omics_agents` importable
try:   # auto-reload edited omics_agents code (local dev only; fails on Colab's old IPython)
    %load_ext autoreload
    %autoreload 2
except ModuleNotFoundError:
    pass
from IPython.display import Markdown, Image, display
import pandas as pd
from omics_agents import config, data, tools
from omics_agents.team import make_team, make_pi

In [ ]:
if config.PROVIDER == "ollama":
    print("Backend: ollama | model:", config.OLLAMA_MODEL, "| url:", config.OLLAMA_BASE_URL)
else:
    assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in the .env file at the repo root (see .env.example)."
    print("Backend: anthropic | model:", config.MODEL, "| effort:", config.EFFORT)

In [ ]:
team = make_team()
pi = make_pi(team)
print("PI tools:", [s["name"] for s in pi.toolbox.schemas])
for k, a in team.items():
    print(f"{k:16s} tools: {[s['name'] for s in a.toolbox.schemas]}  modalities: {a.toolbox.allowed}")

Specialists cannot reach data outside their own layer. For example, the proteomics agent asking for mRNA is refused:

In [ ]:
try:
    team["proteomics"].toolbox.execute("pca_summary", {"modality": "mrna"})
except tools.ToolError as e:
    print("ToolError:", e)

## Run the team

In [ ]:
report = pi.run(
    "Characterise what distinguishes the Basal, Her2 and LumA subtypes across the three omics "
    "layers, check whether the layers tell a consistent biological story, and determine the "
    "best strategy to predict subtype on the held-out test patients.")
display(Markdown(report))

## Cost and communication overhead

In [ ]:
agents = {"PI": pi, **team}
usage = pd.DataFrame({k: {**a.usage, "tool_calls": len(a.tool_log), "usd": round(a.cost(), 4)}
                      for k, a in agents.items()}).T
display(usage)
print(f"Total ~ ${usage['usd'].sum():.3f}")

The PI's tool calls are delegations, and each delegation runs a whole agent loop. This is the extra messaging and coordination overhead of multi-agent systems that the paper discusses under *Limitations of agent architecture*.

### Omics heterogeneity seen in this run
- **Data sparsity:** the protein layer is missing for the test patients, so the integrator had to fall back to the layers that are available.
- **Measurement-error heterogeneity:** layers differ in noise. The mean confidence weights from late fusion show this, since noisier layers get less influence in the vote.
- **Knowledge-representation heterogeneity:** the same biology goes by different names in different layers (gene *ESR1* versus RPPA antibody *ER-alpha*). The PI has to reconcile these names.

In [ ]:
(config.OUTPUT_DIR / "pi_report.md").write_text(report, encoding="utf-8")
all_logs = [{"agent": k, **e} for k, a in agents.items() for e in a.tool_log if e["tool"] != "delegate"]
(config.OUTPUT_DIR / "pi_tool_log.json").write_text(json.dumps(all_logs, indent=1), encoding="utf-8")
print("Saved report + tool log for notebook 04:", len(all_logs), "tool calls")